This notebook might not be usable "out of the box" and is mainly meant to give an idea how the data was calculated from the NMR bundles and the MD ensembles.

# Import modules

In [1]:
import numpy as np
import pandas as pd
import pickle as pkl
import barnaba as bb
from barnaba import definitions, functions
from subprocess import Popen, PIPE
import regex as re
import mdtraj as md
from lib.noe_helper import *

In [2]:
%load_ext autoreload
%autoreload 2
import os
import sys
module_path = os.path.abspath(os.path.join('lib')) # or the path to your source code
sys.path.insert(0, module_path)
from lib.noe_helper import *

# Define Functions

In [3]:
def load( inp ):
    pin = open( inp, "rb" )
    return pkl.load( pin )

def save( outfile, results ):
    with open( outfile + ".pkl", "wb" ) as fp:
        pkl.dump( results, fp )

In [4]:
### HELPER FUNCTIONS TO CALCULATE NOE DISTANCES ###

# helper functions, subsititute strings. This is because the name of hydrogens is a mess
# alt = {"H2'":"1H2'","H5''":"2H5'","H5'":"1H5'","HO2'":"2HO'","H5\"":"2H5'","H5'2":"2H5'","H5'1":"1H5'"}
alt = {"H2'":"H2'1","H5''":"H5'2","H5'":"H5'1","HO2'":"HO'2"}

def sub(ss):
    at = "H" + ss.split("H")[1]
    if(at in alt):
        at = alt[at]
    return ss.split("H")[0] + at

# read experimental datafile and returns a list of labels and experimental values
def read_exp(f_exp):
    labels = []
    vals = []
    with open(f_exp) as fh:
        for line in fh:
            if("#" not in line):
                r1 = line.split()[0].split("-")[0]
                print(r1)
                r2 = line.split()[0].split("-")[1]
                print(r2)
                v1 = np.sort([r1,r2])
                print(v1)
                qq = v1[0] +"/"+ v1[1]
                
                if(qq in labels):
                    print("# DUPLICATE. Skipping data.."),
                    print(qq,vals[labels.index(qq)], line),
                else:
                    vals.append([float(line.split()[1]),float(line.split()[2])])
                    labels.append(qq)
    return labels,vals

# get labels from df column (Assignment) 
def get_labels(df, col):
    labels = []
    for asm in df.iloc[:,col]:
        r1 = asm.split()[0].split("-")[0]
        r2 = asm.split()[0].split("-")[1]
        v1 = np.sort([r1,r2])
        qq = v1[0] +"/"+ v1[1]
        labels.append(qq)
    return labels

# find indeces in topology corresponding to labels in experimental datafile
def get_idxs(labels,top):

    atoms = []
    for atom in top.atoms:
        aa = str(atom).split("-")[1]
        if(aa in alt): aa = alt[aa]
        atoms.append("%s%s" % (str(atom).split("-")[0],aa))
    pairs = []
    for el in labels:
        ss  = el.split("/")
        at1 = sub(ss[0])
        at2 = sub(ss[1])
        if(at1 in atoms and at2 in atoms):
            pairs.append([atoms.index(at1),atoms.index(at2)])
        else:
            print("# Warning: Either %s or %s are missing" % (at1,at2))
            return 0
    if len(pairs) != len(labels):
        print("# Found only %d pairs out of %d" % (len(pairs),len(labels)))
    return np.array(pairs)

def group_by_heading( some_source ):
    buffer= []
    for line in some_source:
        if line.startswith( " ASSI" ):
            if buffer: yield buffer
            buffer= [ line.strip().strip(')') ]
        else:
            buffer.append( line.strip().strip(')') )
    yield buffer

# From NMR bundles

Back-calculate experimental observables from

Since some of the forward models need certain PDB formats we have to reformat the PDB files.

In [5]:
# NMR bundle directory:
#bundles_dir = 'qt_clustering_100/hcp_rdc_3j_theta=16/frames_random_cutoff_0_9'
bundles_dir = 'qt_clustering_100/hcp_rdc_3j_theta=16/frames_random_noReplace_cutoff_0_9'

# Experimental Results directory
exp_data_dir = "exp_data"

# Bundle identifiers:
bundles = []
for i in range(5):
    bundles.append(f"cluster_{i}")
print(bundles)


['cluster_0', 'cluster_1', 'cluster_2', 'cluster_3', 'cluster_4']


## RDCs

We use the pf1-phage prediction method implemented in *PALES** to back-calculate RDCs. In order to do that we use the script ```calc_rdc_bundles.py``` which submits each frame to *PALES*, predicts the alignment tensor and calculates RDCs, parses the output and saves the D values in a Pickle file.

The actual command line to run *PALES is*: ```pales-linux -inD exp_data/exp_rdc_pales.tab -pdb {pdb_tmp} -outD {outd_tmp} -pf1 -H -wv 0.05```


*Zweckstetter, M. NMR: Prediction of molecular alignment from structure using the PALES software. Nat. Protoc. 3, 679–690 (2008).

In [6]:
 # RDCs (works)
rdc_exp_input = pd.read_csv(f'{exp_data_dir}/exp_rdc_pales.tab', names=['RESID_I', 'RESNAME_I', 'ATOMNAME_I', 'RESID_J', 'RESNAME_J',
       'ATOMNAME_J', 'D', 'DD', 'W'], delim_whitespace=True, skiprows=5)
# all residues:
rdc_exp_labels_all = [f"{row[1]['RESNAME_I'][0]}{row[1]['RESID_I']}_{row[1]['ATOMNAME_I']}-{row[1]['ATOMNAME_J']}" for row in rdc_exp_input.iterrows()]
# loop residues:
rdc_exp_labels_loop = [f"{row[1]['RESNAME_I'][0]}{row[1]['RESID_I']}_{row[1]['ATOMNAME_I']}-{row[1]['ATOMNAME_J']}" for row in rdc_exp_input.iterrows() if row[1]['RESID_I'] in [5,6,7,8,9,10]]

In [9]:
for bundle in bundles:
    process = Popen(f'python {bundles_dir}/calc_rdc_bundles.py {bundle} {bundles_dir}', shell=True, stdin=PIPE, stdout=PIPE, universal_newlines=True)
    process.wait() # needed to run one process at a time (because of using temporary files etc.)

Parse the output and write BME readable files:

In [10]:
for bundle in bundles:
    d_df = load(f'{bundles_dir}/nmrbundle_{bundle}_d_df_pf1.pkl')
    for i in rdc_exp_labels_all: # missing residues in 1RNGaxis=1
        if i not in d_df.columns:
            d_df[i] = np.full((len(d_df)), np.nan)
    
    # All res:
    d_df[rdc_exp_labels_all].to_csv(f'{bundles_dir}/nmrbundle_{bundle}_rdc_all_bme.dat', header=False, sep='\t', na_rep=np.nan)
    # Loop only:
    d_df[rdc_exp_labels_loop].to_csv(f'{bundles_dir}/nmrbundle_{bundle}_rdc_loop_bme.dat', header=False, sep='\t', na_rep=np.nan)

The predicted RDCs need to be scaled to the experimental values before comparing them. To reduce the effect of over-fitting we scale the RDCs based on all measurements and since we treat NMR structures as a bundle of structures rather than an ensemble we do the scaling for all individual structures.

In [11]:
for bundle in bundles:
    rdc_bundle = np.loadtxt(f'{bundles_dir}/nmrbundle_{bundle}_rdc_all_bme.dat')
    L = np.zeros(len(rdc_bundle))
    for i in range(len(L)):
        masked = np.ma.array(rdc_bundle[i,1:], mask=np.isnan(rdc_bundle[i,1:])) # mask NaNs (only present in 1RNG due to missing/different residues in the stem)
        L[i] = np.sum(np.array(rdc_exp_input['D'])*masked)/np.sum(masked*masked)
    np.save(f'{bundles_dir}/nmrbundle_{bundle}_rdc_all_bme_L', L)
    rdc_bundle = np.loadtxt(f'{bundles_dir}/nmrbundle_{bundle}_rdc_all_bme.dat')
    L = np.zeros(len(rdc_bundle))
    for i in range(len(L)):
        masked = np.ma.array(rdc_bundle[i,1:], mask=np.isnan(rdc_bundle[i,1:])) # mask NaNs (only present in 1RNG due to missing/different residues in the stem)
        L[i] = np.sum(np.array(rdc_exp_input['D'])*masked)/np.sum(masked*masked)
    np.save(f'{bundles_dir}/nmrbundle_{bundle}_rdc_loop_bme_L', L)

## $^3$J-couplings

We use Barnaba* to calculate the $^3$J scalar couplings 2H5H4, H1H2, H2H3, H3P, C4Pb, 1H5P, 1H5H4, C4Pe, 2H5P and H3H4.

Definition: $A cos^2 (\theta + \phi) + B cos (\theta + \phi) + C$

*Bottaro, S. et al. Barnaba: software for analysis of nucleic acid structures and trajectories. Rna 25, 219–231 (2019)

In [6]:
# A (Hz), B (Hz), C (Hz), _, phi (rad)
bb.definitions.couplings_karplus

{'H1H2': [9.67, -2.03, 0.0, 0.0, 0.0],
 'H2H3': [9.67, -2.03, 0.0, 0.0, 0.0],
 'H3H4': [9.67, -2.03, 0.0, 0.0, 0.0],
 '1H5P': [15.3, -6.1, 1.6, 0.0, -2.094395],
 '2H5P': [15.3, -6.1, 1.6, 0.0, 2.094395],
 'C4Pb': [6.9, -3.4, 0.7, 0.0, 0.0],
 '1H5H4': [9.7, -1.8, 0.0, 0.0, -2.094395],
 '2H5H4': [9.7, -1.8, 0.0, 0.0, 0.0],
 'H3P': [15.3, -6.1, 1.6, 0.0, 2.094395],
 'C4Pe': [6.9, -3.4, 0.7, 0.0, 0.0],
 'H1C2/4': [4.7, 2.3, 0.1, 0.0, -1.0471975],
 'H1C6/8': [4.5, -0.6, 0.1, 0.0, -1.0471975]}

In [11]:
ll = [10.974727630615234,0.06831999868154526,0.8671687841415405,8.968681335449219,10.989727020263672,6.426058769226074,2.143512725830078,4.50574254989624,8.193836212158203,8.603635787963867,7.94950008392334,10.064437866210938,10.070903778076172,8.552502632141113,8.070211410522461,8.814104080200195,2.5965662002563477,1.6359444856643677,1.7245982885360718,4.839846134185791,7.233496189117432,2.166797161102295,3.3240880966186523,3.1809096336364746,1.0627166032791138,1.3175597190856934]
print(len(ll))

26


In [12]:
j3_exp_labels = list(pd.read_csv('exp_data/exp_j3_bme.dat', delim_whitespace=True).index) # ONLY EXTENDED-LOOP RESIDUES!!!
j3_exp_couplings = list(set([ i.split('-')[-1] for i in j3_exp_labels ]))

# Never used?
barnaba_to_exp = {'H1H2':"H1',H2'", 'H2H3':"H2',H3'", 'H3H4':"H3',H4'",\
                '1H5P':"H5'i,Pi", '2H5P':"H5''i,Pi",\
                'C4Pb':"C4'i,Pi", '1H5H4':"NA", '2H5H4':"NA", \
                'H3P':"H3'i,Pi+1" ,'C4Pe':"C4'i,Pi+1", \
                'H1C2/4':"NA", 'H1C6/8':"NA"}

# the shape of couplings is (nframes, nresidues, ncouplings)
# only look at bundle A
for bundle in bundles:
    traj_file   = f'{bundles_dir}/{bundle}_ccr.pdb'
    top_file    = f'{bundles_dir}/{bundle}_ccr.pdb'

    couplings,residues = bb.jcouplings(traj_file, topology=top_file, couplings=j3_exp_couplings)
    print( f'Calculated jcouplings for {j3_exp_couplings}.' )
    print( couplings.shape )

    couplings_avg = np.average( couplings, 0 )
    print(couplings_avg.shape)
    
    j3_calc_ = pd.DataFrame()
    for i,ii in enumerate([ i.split('_')[0]+i.split('_')[1] for i in residues]):
        for j,jj in enumerate(j3_exp_couplings):
            print(f'Bundle {traj_file} Coupling: {ii}-{jj}')
            j3_calc_[f'{ii}-{jj}'] = couplings[:,i,j]

    ll = []
    for i in j3_exp_labels:
        ll.append(j3_calc_[i])
    pd.DataFrame(ll).T.to_csv(f'{bundles_dir}/nmrbundle_{bundle}_j3_loop_bme.dat', sep=' ', header=False)

# Loading qt_clustering_100/hcp_rdc_3j_theta=16/frames_random_noReplace_cutoff_0_9/cluster_0_ccr.pdb 


Calculated jcouplings for ['C4Pe', 'H3P', 'H2H3', 'H1H2', '1H5P', '2H5P', 'H3H4'].
(33, 14, 7)
(14, 7)
Bundle qt_clustering_100/hcp_rdc_3j_theta=16/frames_random_noReplace_cutoff_0_9/cluster_0_ccr.pdb Coupling: G1-C4Pe
Bundle qt_clustering_100/hcp_rdc_3j_theta=16/frames_random_noReplace_cutoff_0_9/cluster_0_ccr.pdb Coupling: G1-H3P
Bundle qt_clustering_100/hcp_rdc_3j_theta=16/frames_random_noReplace_cutoff_0_9/cluster_0_ccr.pdb Coupling: G1-H2H3
Bundle qt_clustering_100/hcp_rdc_3j_theta=16/frames_random_noReplace_cutoff_0_9/cluster_0_ccr.pdb Coupling: G1-H1H2
Bundle qt_clustering_100/hcp_rdc_3j_theta=16/frames_random_noReplace_cutoff_0_9/cluster_0_ccr.pdb Coupling: G1-1H5P
Bundle qt_clustering_100/hcp_rdc_3j_theta=16/frames_random_noReplace_cutoff_0_9/cluster_0_ccr.pdb Coupling: G1-2H5P
Bundle qt_clustering_100/hcp_rdc_3j_theta=16/frames_random_noReplace_cutoff_0_9/cluster_0_ccr.pdb Coupling: G1-H3H4
Bundle qt_clustering_100/hcp_rdc_3j_theta=16/frames_random_noReplace_cutoff_0_9/cluste

# Loading qt_clustering_100/hcp_rdc_3j_theta=16/frames_random_noReplace_cutoff_0_9/cluster_1_ccr.pdb 


Calculated jcouplings for ['C4Pe', 'H3P', 'H2H3', 'H1H2', '1H5P', '2H5P', 'H3H4'].
(15, 14, 7)
(14, 7)
Bundle qt_clustering_100/hcp_rdc_3j_theta=16/frames_random_noReplace_cutoff_0_9/cluster_1_ccr.pdb Coupling: G1-C4Pe
Bundle qt_clustering_100/hcp_rdc_3j_theta=16/frames_random_noReplace_cutoff_0_9/cluster_1_ccr.pdb Coupling: G1-H3P
Bundle qt_clustering_100/hcp_rdc_3j_theta=16/frames_random_noReplace_cutoff_0_9/cluster_1_ccr.pdb Coupling: G1-H2H3
Bundle qt_clustering_100/hcp_rdc_3j_theta=16/frames_random_noReplace_cutoff_0_9/cluster_1_ccr.pdb Coupling: G1-H1H2
Bundle qt_clustering_100/hcp_rdc_3j_theta=16/frames_random_noReplace_cutoff_0_9/cluster_1_ccr.pdb Coupling: G1-1H5P
Bundle qt_clustering_100/hcp_rdc_3j_theta=16/frames_random_noReplace_cutoff_0_9/cluster_1_ccr.pdb Coupling: G1-2H5P
Bundle qt_clustering_100/hcp_rdc_3j_theta=16/frames_random_noReplace_cutoff_0_9/cluster_1_ccr.pdb Coupling: G1-H3H4
Bundle qt_clustering_100/hcp_rdc_3j_theta=16/frames_random_noReplace_cutoff_0_9/cluste

# Loading qt_clustering_100/hcp_rdc_3j_theta=16/frames_random_noReplace_cutoff_0_9/cluster_2_ccr.pdb 
# Loading qt_clustering_100/hcp_rdc_3j_theta=16/frames_random_noReplace_cutoff_0_9/cluster_3_ccr.pdb 


Calculated jcouplings for ['C4Pe', 'H3P', 'H2H3', 'H1H2', '1H5P', '2H5P', 'H3H4'].
(10, 14, 7)
(14, 7)
Bundle qt_clustering_100/hcp_rdc_3j_theta=16/frames_random_noReplace_cutoff_0_9/cluster_2_ccr.pdb Coupling: G1-C4Pe
Bundle qt_clustering_100/hcp_rdc_3j_theta=16/frames_random_noReplace_cutoff_0_9/cluster_2_ccr.pdb Coupling: G1-H3P
Bundle qt_clustering_100/hcp_rdc_3j_theta=16/frames_random_noReplace_cutoff_0_9/cluster_2_ccr.pdb Coupling: G1-H2H3
Bundle qt_clustering_100/hcp_rdc_3j_theta=16/frames_random_noReplace_cutoff_0_9/cluster_2_ccr.pdb Coupling: G1-H1H2
Bundle qt_clustering_100/hcp_rdc_3j_theta=16/frames_random_noReplace_cutoff_0_9/cluster_2_ccr.pdb Coupling: G1-1H5P
Bundle qt_clustering_100/hcp_rdc_3j_theta=16/frames_random_noReplace_cutoff_0_9/cluster_2_ccr.pdb Coupling: G1-2H5P
Bundle qt_clustering_100/hcp_rdc_3j_theta=16/frames_random_noReplace_cutoff_0_9/cluster_2_ccr.pdb Coupling: G1-H3H4
Bundle qt_clustering_100/hcp_rdc_3j_theta=16/frames_random_noReplace_cutoff_0_9/cluste

# Loading qt_clustering_100/hcp_rdc_3j_theta=16/frames_random_noReplace_cutoff_0_9/cluster_4_ccr.pdb 


## CCRs

We use the script ```calc_ccr.py``` to back-calculate CCRs from pdb files or trajectories
Important notes:
- The PDB naming has to be in a specific format (check the script if in doubt).
- Here, $\Gamma$-HCP (C4p-P, C3p-P-plus, C4p-P-plus) and $\Gamma$-HCNCH (C1-CC) are measured at 600 MHz (14.09 T) while $\Gamma$-HCCH (C1-C2, C3-C4) is measured at 700 MHz (16.44 T) which means that in theory the script has to be executed twice and B0 has to be adjusted accordingly. However, $\Gamma$-HCCH couplings are not influenced by B0 which is why we can ignore this for now.

Calculate CCRs from bundles:

Note: certain atoms need specific naming in the .pdb files. Change manualy if necessary 

In [19]:
# make sure the pdbs use OP1 and OP2 instead of O1P and O2P
ccrs = {"HCCH":14.09,
        "HCN":16.44,
        "HCP":22.30}

for ccr in ccrs:
    field = ccrs[ccr]
    # Check the script and change output name etc.
    for bundle in bundles:
        process = Popen(f'python {bundles_dir}/calc_ccr_2.py {bundles_dir}/{bundle}_ccr.pdb "{field}"', shell=True, stdin=PIPE, stdout=PIPE, universal_newlines=True)
        process.wait() # needed to run one process at a time (because of using temporary files etc.

In [21]:
for ccr in ccrs:
    field = ccrs[ccr]
    ccr_exp_labels_loop  = list(pd.read_csv(f'exp_data/exp_ccr_{ccr}.dat', delim_whitespace=True, usecols=[0]).index)
    ccr_exp_labels_loop_2F87  = list(pd.read_csv(f'exp_data/exp_ccr_{ccr}_2F87_loop_bme.dat', delim_whitespace=True, usecols=[0]).index)
    for bundle in bundles:
        tmp_ccr = pd.read_csv(f'{bundles_dir}/{bundle}_ccr_{field}_calc.dat', delim_whitespace=True, index_col=0)
        tmp_ccr_loop = tmp_ccr[ccr_exp_labels_loop]
        # Check if all data points are contained in the bundle:
        print(list(tmp_ccr_loop.columns))
        # Save as BME readable file:
        tmp_ccr_loop.to_csv(f'{bundles_dir}/nmrbundle_{bundle}_ccr_calc_{ccr}_loop_bme.dat', header=False, sep=' ')

['C5:C1-C2', 'G6:C1-C2', 'A7:C1-C2', 'A8:C1-C2', 'G9:C1-C2', 'G10:C1-C2', 'C5:C3-C4', 'G6:C3-C4', 'A7:C3-C4', 'A8:C3-C4', 'G9:C3-C4', 'G10:C3-C4']
['C5:C1-C2', 'G6:C1-C2', 'A7:C1-C2', 'A8:C1-C2', 'G9:C1-C2', 'G10:C1-C2', 'C5:C3-C4', 'G6:C3-C4', 'A7:C3-C4', 'A8:C3-C4', 'G9:C3-C4', 'G10:C3-C4']
['C5:C1-C2', 'G6:C1-C2', 'A7:C1-C2', 'A8:C1-C2', 'G9:C1-C2', 'G10:C1-C2', 'C5:C3-C4', 'G6:C3-C4', 'A7:C3-C4', 'A8:C3-C4', 'G9:C3-C4', 'G10:C3-C4']
['C5:C1-C2', 'G6:C1-C2', 'A7:C1-C2', 'A8:C1-C2', 'G9:C1-C2', 'G10:C1-C2', 'C5:C3-C4', 'G6:C3-C4', 'A7:C3-C4', 'A8:C3-C4', 'G9:C3-C4', 'G10:C3-C4']
['C5:C1-C2', 'G6:C1-C2', 'A7:C1-C2', 'A8:C1-C2', 'G9:C1-C2', 'G10:C1-C2', 'C5:C3-C4', 'G6:C3-C4', 'A7:C3-C4', 'A8:C3-C4', 'G9:C3-C4', 'G10:C3-C4']
['C5:C1-CC', 'A7:C1-CC', 'A8:C1-CC', 'G9:C1-CC', 'G10:C1-CC']
['C5:C1-CC', 'A7:C1-CC', 'A8:C1-CC', 'G9:C1-CC', 'G10:C1-CC']
['C5:C1-CC', 'A7:C1-CC', 'A8:C1-CC', 'G9:C1-CC', 'G10:C1-CC']
['C5:C1-CC', 'A7:C1-CC', 'A8:C1-CC', 'G9:C1-CC', 'G10:C1-CC']
['C5:C1-CC', 'A7:

Parse calculated data and write to BME file:

In [55]:
#for bundle in bundles:
#    tmp_ccr_600 = pd.read_csv(f'{bundles_dir}/ccr/600MHz/Bundle_{bundle}_gmx_ccr_calc.dat', delim_whitespace=True, index_col=0)
#    tmp_ccr_loop_600 = tmp_ccr_600[ccr_exp_labels_loop]
#    tmp_ccr_700 = pd.read_csv(f'{bundles_dir}/ccr/700MHz/Bundle_{bundle}_gmx_ccr_calc.dat', delim_whitespace=True, index_col=0)
#    # Check if all data points are contained in the bundle:
#    print(list(tmp_ccr_loop.columns) == ccr_lbl)
#    # Combine 700 MHz data for gamma-HCCH (C1-C2, C3-C4) couplings with the rest at 600 MHz:
#    for lbl in [ i for i in ccr_exp_labels_loop if i.split(':')[1] in ['C1-C2', 'C3-C4'] ]:
#        tmp_ccr_600[lbl] = tmp_ccr_700[lbl]
#    # Save as BME readable file:
#    tmp_ccr_loop_600.to_csv(f'{bundles_dir}/ccr/Bundle_{bundle}_gmx_ccr_calc_loop_bme.dat', header=False, sep=' ')

NOTE: Don't forget about adding nans for missing measurements in 1RNG when compiling a data file for all residues (including stem).

## NOEs

We load NOEs from an ARIA output file after they have been converted to distances (\AA). For this, we use the NOEs from bundle F because those have been restraint the most by other types of data. Generally, they don't differ significantly between the different bundles.

Load and parse experimental NOE distance file:

In [ ]:
seq = {1:'G', 2:'G', 3:'C', 4:'A', 5:'C', 6:'G', 7:'A', 8:'A', 9:'G', 10:'G', 11:'U', 12:'G', 13:'C', 14:'C'}
for bundle in bundles:    
    df_noe_exp = pd.DataFrame()
    
    with open(f'exp_data/exp_noe.tbl') as f: # use NOEs from bundle 2+2
        
        for heading_and_lines in group_by_heading( f ):
            heading= heading_and_lines[0]
            lines= heading_and_lines[1:]
            
            resid1 = int(re.search('resid (\d+)', lines[0] ).group(1))
            name1 = re.search('name (.+)', lines[0] ).group(1).strip()
            resid2 = int(re.search('resid (\d+)', lines[1] ).group(1))
            name2 = re.search('name (.+)', lines[1] ).group(1).strip()
            
            row = {'Assignment': f'{seq[resid1]}{resid1}{name1}-{seq[resid2]}{resid2}{name2}', 'distance':float(lines[2].split()[0]), 'upper_err':float(lines[2].split()[1]), 'lower_err':float(lines[2].split()[2])}
            df_row = pd.DataFrame.from_dict(row, orient='index')
            df_noe_exp = pd.concat([df_noe_exp, df_row], axis=1)
        df_noe_exp = df_noe_exp.T

295
295
295
295
295


Make separate dfs for loop NOEs:

In [14]:
residues = ['G1', 'G2', 'C3', 'A4', 'C5', 'G6', 'A7', 'A8', 'G9', 'G10', 'U11', 'G12', 'C13', 'C14']
loop_c5 = ['C5','G6', 'A7', 'A8', 'G9','G10']


def create_df_noe_exp_loop( loop):
    a_loop = []
    for i,n in df_noe_exp[:].iterrows():
        a = n['Assignment'].split('-')
        r1 = re.split('(^[A-Z]\d+)', a[0])[1]
        r2 = re.split('(^[A-Z]\d+)', a[1])[1]

        if r1 in loop and r2 in loop:
            a_loop.append(n['Assignment'])
            print(f"Added Assignment: {n['Assignment']}")

        # elif r1 in loop or r2 in loop:
        #     a_both.append(n['Assignment'])
        # else:
        #     a_stem.append(n['Assignment'])
    print(f'Loop NOEs: {len(a_loop)}')
    # print( f'Loop NOEs: {len(a_loop)}\nStem NOEs: {len(a_stem)}\nIntersect: {len(a_both)}' )
    return df_noe_exp.loc[df_noe_exp.Assignment.isin(a_loop)], a_loop

df_noe_exp_loop, a_loop = create_df_noe_exp_loop(loop_c5)

print("shape",df_noe_exp_loop.shape)

Added Assignment: A7H5'1-A8H8
Added Assignment: A7H2'-A8H8
Added Assignment: C5H5-C5H1'
Added Assignment: C5H5'2-C5H1'
Added Assignment: C5H5'2-C5H5
Added Assignment: G6H3'-G6H1'
Added Assignment: G6H5'1-G6H1'
Added Assignment: G6H5'2-G6H1'
Added Assignment: A7H2'-A7H1'
Added Assignment: A7H5'2-A7H1'
Added Assignment: A7H5'2-A7H5'1
Added Assignment: A8H5'2-A8H1'
Added Assignment: G9H2'-G9H3'
Added Assignment: G9H3'-A8H2
Added Assignment: G9H4'-G9H3'
Added Assignment: G9H5'1-G9H3'
Added Assignment: G10H4'-G9H1'
Added Assignment: A7H4'-A7H1'
Added Assignment: A7H2'-A7H8
Added Assignment: C5H1'-G6H1'
Added Assignment: C5H2'-C5H5
Added Assignment: C5H3'-C5H6
Added Assignment: G6H1'-A7H8
Added Assignment: G6H3'-A7H8
Added Assignment: G6H5'1-A7H8
Added Assignment: A7H5'1-A8H8
Added Assignment: A7H5'2-G6H1'
Added Assignment: A8H4'-A7H1'
Added Assignment: A8H4'-A7H2
Added Assignment: A8H5'1-A8H1'
Added Assignment: A8H5'2-A8H5'1
Added Assignment: A8H5'2-G9H8
Added Assignment: G9H3'-A8H1'
Added 

In [31]:
ll = [6.9179344,3.7633753,5.1222486,4.396636,4.1226954,3.8597145,4.928258,4.9345117,2.9290776,5.234475,1.7964973,5.291297,2.3039763,9.484493,2.9690742,3.946378,4.747525,3.7402935,2.0299027,7.2778883,5.814543,3.1696358,6.647728,5.7001114,8.425046,6.9179344,4.4419804,7.144528,11.752095,4.610272,1.7252831,6.6752915,6.8026257,3.779589,5.1521926,2.9116485,1.7592608,4.607394,4.9127107,2.2306285,4.4513493,6.919407,5.9012036,5.242957,4.0893173,6.776887,7.385356,7.037287,3.465907,5.943649,4.018281,3.5382032,2.7445064,4.2702885,5.0623617,3.7346604,2.2817113,2.3251052,5.38703,3.0723605,4.0920644,4.465981,3.669454,1.8064624,4.7034974,6.255725,4.1784444,4.3275976,11.389121,2.7807367,2.4116597,4.1909575,2.0997157,4.8179884,2.6000934,1.7242961,5.2828884,3.623178,5.366695,3.826538,2.2114487,4.489842,3.75955,7.4617558,3.0858388,8.085283,6.893518,13.690848,8.280705,10.273857,9.204475,5.6412325,4.6921663,4.2951136,7.4364834,3.751187,3.6193933,3.3054638,4.768954,4.55424,4.528677,8.71023,10.990043,4.363311,3.821662,5.324669,9.8064575,12.655135,2.6192749,6.2369924,2.9217293,3.8258364,2.1704621,3.896473,8.084484,3.5558553,4.386138,6.81964,6.7816377,10.593187,4.43966,14.992031,6.5077953,3.836273,2.6752386,3.8113256,8.07433,9.518562,3.2942986,7.028093,16.429474,9.895144,4.7360425,10.693441,8.982596]
print(len(ll))

135


In [34]:
# Calc distances from the bundles (1ZIH has to be one separately due to missing/different residues)
for bundle in bundles: #all except last 1ZIH
    print(bundle)
    traj_ = md.load_pdb(f'{bundles_dir}/{bundle}_ccr.pdb')
    # labels = get_labels(df_noe_exp, 0) # For all NOEs
    labels = get_labels(df_noe_exp_loop, 0)
    pairs = get_idxs( labels, traj_.topology )
    #print(f'labels:\n{labels}')
    #print(f'pairs:\n{pairs}')
    # calculate distances multiply by 10 to convert to angs
    dists = 10.0*md.compute_distances(traj_,pairs)
    df_noe = pd.DataFrame(dists)
    
    df_noe.columns = list(df_noe_exp_loop.Assignment)
    # Save to BME format:
    df_noe.to_csv(f'{bundles_dir}/nmrbundle_{bundle}_dists_loop.dat', sep='\t', header=False)
    print(df_noe.shape)

cluster_0
(33, 135)
cluster_1
(15, 135)
cluster_2
(10, 135)
cluster_3
(6, 135)
cluster_4
(5, 135)
